In [0]:
from pyspark.sql.functions import col

files = [
    "Product", "Region", "Reseller", "Sales_2017", 
    "Sales_2018", "Sales_2019", "Sales_2020", 
    "Salesperson", "SalespersonRegion", "Targets"
]

base_path = "/Volumes/workspace/bronze/raw_data/"

for file_name in files:
    print(f"Processing: {file_name}...")
    
    # 1. READ CSV - delimiter (Tab)
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("sep", "\t") \
        .load(f"{base_path}{file_name}.csv")
    
    # 2. FIX COLUMN NAMES
    for col_name in df.columns:
        clean_name = col_name.replace(" ", "_") \
                             .replace("\t", "_") \
                             .replace("\n", "_") \
                             .replace("-", "_") 
    
        
        df = df.withColumnRenamed(col_name, clean_name)
    
    # 3. SAVE AS DELTA TABLE
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"bronze.{file_name.lower()}")

print("--- Bronze Layer Built successfully! ---")